In [2]:
from transformers import TimesFmModel, TimesFmConfig
import torch
import numpy as np

In [8]:
model_id = "google/timesfm-2.0-500m-pytorch"
model = TimesFmModel.from_pretrained(model_id)
model.eval()

Loading weights: 0it [00:00, ?it/s]
TimesFmModel LOAD REPORT from: google/timesfm-2.0-500m-pytorch
Key                                             | Status     | 
------------------------------------------------+------------+-
decoder.layers.{0...49}.self_attn.k_proj.bias   | UNEXPECTED | 
decoder.layers.{0...49}.mlp.down_proj.bias      | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.k_proj.weight | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.v_proj.bias   | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.q_proj.weight | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.o_proj.weight | UNEXPECTED | 
decoder.layers.{0...49}.mlp.layer_norm.bias     | UNEXPECTED | 
decoder.layers.{0...49}.mlp.gate_proj.weight    | UNEXPECTED | 
decoder.layers.{0...49}.mlp.gate_proj.bias      | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.scaling       | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.v_proj.weight | UNEXPECTED | 
decoder.layers.{0...49}.mlp.layer_norm.weight   | UNEXPECTED | 
decod

TimesFmModel(
  (input_ff_layer): TimesFmResidualBlock(
    (input_layer): Linear(in_features=64, out_features=1280, bias=True)
    (activation): SiLU()
    (output_layer): Linear(in_features=1280, out_features=1280, bias=True)
    (residual_layer): Linear(in_features=64, out_features=1280, bias=True)
  )
  (freq_emb): Embedding(3, 1280)
  (layers): ModuleList(
    (0-49): 50 x TimesFmDecoderLayer(
      (self_attn): TimesFmAttention(
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (o_proj): Linear(in_features=1280, out_features=1280, bias=True)
      )
      (mlp): TimesFmMLP(
        (gate_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (down_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (layer_norm): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      )

In [17]:
batch_size = 100
seq_len = 512 

device = torch.device("mps")
model = model.to(device)

vibration_data = torch.randn(batch_size, seq_len).to(device)

past_values_padding = torch.zeros(batch_size, seq_len, dtype=torch.long).to(device)
freq = torch.zeros(batch_size, dtype=torch.long).view(-1, 1).to(device)

with torch.no_grad():
    outputs = model(
        past_values=vibration_data,
        past_values_padding=past_values_padding,
        freq=freq
    )

embeddings = outputs.last_hidden_state

vibration_fingerprints = (
    torch.mean(embeddings, dim=1)
    .detach()
    .cpu()
    .numpy()
)

print(f"Final shape for clustering: {vibration_fingerprints.shape}")

Final shape for clustering: (100, 1280)
